In [1]:
# ============================================================
# Notebook    : 10_governance_fairness_case_a.ipynb
# Description : Fairness audit for Case A (③c LightGBM-Aggregate),
#               using VehType as a PROXY cohort — replacing v1's
#               flawed "cohort = observation-start-year" design.
#
#               IMPORTANT FRAMING DIFFERENCE from notebook 11:
#               VehType is NOT a protected attribute — it's a
#               vehicle characteristic. A gap across VehType
#               groups does not by itself mean "unfair" the way
#               a GENDER/RACE gap would. The question here is
#               narrower: "does this model produce systematically
#               worse errors for people who happen to own certain
#               vehicle types?" — a signal worth monitoring, not
#               a protected-class violation to correct.
#
#               All 16 VehType categories shown (per design
#               decision) — small-n groups flagged explicitly
#               rather than summarized away.
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install lightgbm pandas numpy matplotlib


# ============================================================
# 1. Common imports
# ============================================================
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

np.random.seed(42)


# ============================================================
# 2. Reconstruct the EXACT ③c test set (same as notebooks 03c
#    and 09) — same source CSV, same split_idpols.json
# ============================================================
df = pd.read_csv("data/fremotor_multi_history_aggregate.csv")

with open("data/sequences/split_idpols.json", "r", encoding="utf-8") as f:
    split_ids = json.load(f)
test_idpols = set(split_ids["test"])

df_test = df[df["IDpol"].isin(test_idpols)].copy()

NUMERIC_COLS = [
    "Expo", "YearGap", "CumClaimCount", "ClaimRateSoFar", "YearsSinceFirstClaimLog",
]
BINARY_COLS  = ["PrevLabel", "HasPriorClaim"]
CAT_COLS     = ["Usage", "VehType", "VehPower"]
FEATURE_COLS = NUMERIC_COLS + BINARY_COLS + CAT_COLS

for col in CAT_COLS:
    df_test[col] = df_test[col].astype("category")

X_test = df_test[FEATURE_COLS].copy()
y_test = df_test["Label"].copy()

print(f"[CHECK 1] Test set: {X_test.shape}")
print(f"[CHECK 1] Positive rate: {y_test.mean()*100:.2f}%")


# ============================================================
# 3. Load model, predict
# ============================================================
model = lgb.Booster(model_file="data/sequences/lightgbm_aggregate_model.txt")
y_pred_prob = model.predict(X_test)
y_pred_cls  = (y_pred_prob >= 0.5).astype(int)

print(f"\n[CHECK 2] Predicted positive rate: {y_pred_cls.mean()*100:.2f}%")


# ============================================================
# 4. VehType distribution — full 16-category breakdown, flagging
#    small-n groups (threshold: n < 100 as a rough noisiness cue,
#    matching the order of magnitude where Case B's RACE minority
#    (n=162) already required an uncertainty caveat)
# ============================================================
vehtype_counts = df_test["VehType"].value_counts().sort_index()
print(f"\n[CHECK 3] VehType distribution in test set (16 categories):")
print(vehtype_counts.to_string())

small_n_types = vehtype_counts[vehtype_counts < 100].index.tolist()
if small_n_types:
    print(f"\n[CHECK 3] WARNING — VehType groups with n<100 "
          f"(treat metrics as noisy): {small_n_types}")


# ============================================================
# 5. Metric function — same three-metric structure as notebook
#    11, generalized to N groups instead of 2
# ============================================================
def compute_group_metrics(y_true, y_pred_cls, y_pred_prob, group_labels):
    results = []
    for g in sorted(pd.Series(group_labels).unique()):
        mask = (group_labels == g)
        yt = y_true[mask]
        yp = y_pred_cls[mask]
        ypp = y_pred_prob[mask]

        if mask.sum() == 0:
            continue

        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

        results.append({
            "VehType": g,
            "n": mask.sum(),
            "actual_positive_rate": yt.mean(),
            "predicted_positive_rate": yp.mean(),
            "TPR": tpr,
            "FPR": fpr,
            "mean_pred_prob": ypp.mean(),
        })
    return pd.DataFrame(results)


# ============================================================
# 6. Run audit — all 16 VehType groups
# ============================================================
print("\n" + "="*60)
print("VehType — Fairness Audit (16 groups, proxy cohort)")
print("="*60)

vehtype_metrics = compute_group_metrics(
    y_test.values, y_pred_cls, y_pred_prob, df_test["VehType"].values
)
vehtype_metrics = vehtype_metrics.merge(
    pd.DataFrame({"VehType": small_n_types, "small_n_flag": True}),
    on="VehType", how="left"
)
vehtype_metrics["small_n_flag"] = vehtype_metrics["small_n_flag"].fillna(False)

print("\n[VehType] Full group metrics (all 16 categories):")
print(vehtype_metrics.to_string(index=False))


# ============================================================
# 7. Gap summary — computed BOTH on all 16 groups and on
#    "reliable" groups only (n>=100), since a single small-n
#    outlier can dominate a max-min gap misleadingly
# ============================================================
def gap_summary(df_metrics, label):
    dp_gap  = df_metrics["predicted_positive_rate"].max() - df_metrics["predicted_positive_rate"].min()
    tpr_gap = df_metrics["TPR"].max() - df_metrics["TPR"].min()
    fpr_gap = df_metrics["FPR"].max() - df_metrics["FPR"].min()
    print(f"\n[{label}] DP gap: {dp_gap:.4f} | TPR gap: {tpr_gap:.4f} | FPR gap: {fpr_gap:.4f}")
    return dp_gap, tpr_gap, fpr_gap

print("\n" + "-"*60)
dp_all, tpr_all, fpr_all = gap_summary(vehtype_metrics, "ALL 16 groups")

reliable = vehtype_metrics[~vehtype_metrics["small_n_flag"]]
print(f"\n[Reliable groups only] n>=100, {len(reliable)}/{len(vehtype_metrics)} groups retained")
dp_rel, tpr_rel, fpr_rel = gap_summary(reliable, "RELIABLE groups (n>=100)")

# which group drives the gap, and is it a small-n group?
worst_tpr_group = vehtype_metrics.loc[vehtype_metrics["TPR"].idxmin()]
best_tpr_group  = vehtype_metrics.loc[vehtype_metrics["TPR"].idxmax()]
print(f"\n[VehType] Lowest TPR : {worst_tpr_group['VehType']} "
      f"(TPR={worst_tpr_group['TPR']:.4f}, n={worst_tpr_group['n']}, "
      f"small_n={worst_tpr_group['small_n_flag']})")
print(f"[VehType] Highest TPR: {best_tpr_group['VehType']} "
      f"(TPR={best_tpr_group['TPR']:.4f}, n={best_tpr_group['n']}, "
      f"small_n={best_tpr_group['small_n_flag']})")


# ============================================================
# 8. Visualization
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

x = np.arange(len(vehtype_metrics))
colors = ["tab:red" if flag else "tab:blue" for flag in vehtype_metrics["small_n_flag"]]

axes[0].bar(x, vehtype_metrics["TPR"], color=colors)
axes[0].set_xticks(x)
axes[0].set_xticklabels(vehtype_metrics["VehType"], rotation=45, ha="right")
axes[0].set_title("TPR by VehType (red = n<100, treat as noisy)")
axes[0].set_ylabel("TPR")

axes[1].bar(x, vehtype_metrics["FPR"], color=colors)
axes[1].set_xticks(x)
axes[1].set_xticklabels(vehtype_metrics["VehType"], rotation=45, ha="right")
axes[1].set_title("FPR by VehType (red = n<100, treat as noisy)")
axes[1].set_ylabel("FPR")

plt.tight_layout()
plt.savefig("data/sequences/fairness_case_a_vehtype.png")
plt.close()
print("\nSaved: data/sequences/fairness_case_a_vehtype.png")


# ============================================================
# 9. Save
# ============================================================
vehtype_metrics.to_csv("data/sequences/fairness_case_a_vehtype_metrics.csv", index=False)
print("Saved: data/sequences/fairness_case_a_vehtype_metrics.csv")


# ============================================================
# 10. Summary
# ============================================================
print("\n===== Case A Fairness Audit Summary (VehType proxy cohort) =====")
print(f"Groups total          : {len(vehtype_metrics)}")
print(f"Small-n groups (n<100): {len(small_n_types)} — {small_n_types}")
print(f"All-groups gaps       : DP={dp_all:.4f}, TPR={tpr_all:.4f}, FPR={fpr_all:.4f}")
print(f"Reliable-groups gaps  : DP={dp_rel:.4f}, TPR={tpr_rel:.4f}, FPR={fpr_rel:.4f}")
print(f"\nNOTE: VehType is a PROXY cohort, not a protected attribute.")
print(f"A gap here signals uneven model error across vehicle types,")
print(f"not a protected-class fairness violation. Recall from notebook 09's")
print(f"SHAP: VehType was Case A's #2 feature (mean|SHAP|=0.325) — the")
print(f"model relies on it heavily, so uneven errors across its categories")
print(f"are worth monitoring even though this isn't a protected-attribute audit.")
print("===================================================================")

[CHECK 1] Test set: (54755, 10)
[CHECK 1] Positive rate: 12.79%

[CHECK 2] Predicted positive rate: 41.29%

[CHECK 3] VehType distribution in test set (16 categories):
VehType
VehType1         7
VehType10    39315
VehType11     2635
VehType12      843
VehType13      120
VehType14       30
VehType15       23
VehType2        47
VehType3      1550
VehType4        52
VehType5       220
VehType6      3118
VehType7      1328
VehType8      3545
VehType9      1922

[CHECK 3] WARNING — VehType groups with n<100 (treat metrics as noisy): ['VehType1', 'VehType14', 'VehType15', 'VehType2', 'VehType4']

VehType — Fairness Audit (16 groups, proxy cohort)


C:\Users\User\AppData\Local\Temp\ipykernel_9052\3277739600.py:142: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vehtype_metrics["small_n_flag"] = vehtype_metrics["small_n_flag"].fillna(False)



[VehType] Full group metrics (all 16 categories):
  VehType     n  actual_positive_rate  predicted_positive_rate      TPR      FPR  mean_pred_prob  small_n_flag
 VehType1     7              0.000000                 0.000000      NaN 0.000000        0.097344          True
VehType10 39315              0.149612                 0.500878 0.848011 0.439805        0.449809         False
VehType11  2635              0.082732                 0.229222 0.559633 0.199421        0.314869         False
VehType12   843              0.060498                 0.124555 0.254902 0.116162        0.256740         False
VehType13   120              0.025000                 0.000000 0.000000 0.000000        0.102931         False
VehType14    30              0.166667                 0.233333 0.600000 0.160000        0.334840          True
VehType15    23              0.304348                 0.565217 0.714286 0.500000        0.515780          True
 VehType2    47              0.042553                 0.08510